# LoRA fine-tuning Qwen3.5-4B на Text2SQL

Полный пайплайн: install → load model → train LoRA → merge → GGUF → download.

**Окружение:** Google Colab, Tesla T4 (16 GB VRAM) — бесплатный tier.

**Входные данные** (загрузить через `Files` ВРУЧНУЮ или подключить Google Drive):
- `train.jsonl` — выход `scripts/build_finetune_dataset.py`
- `val.jsonl` — то же

**Учебные нюансы по ходу:**
- Почему unsloth, а не plain peft.
- Зачем `apply_chat_template` именно от tokenizer'a, а не вручную.
- Как читать loss-кривую: что есть здоровый train, что overfit.
- GGUF-конвертация: почему q4_K_M по умолчанию.

**Время прогона:** ~40-60 мин на ~5k примеров за 2 эпохи.

## 0. Установка зависимостей

**Учебный нюанс:** `unsloth` — это уровень абстракции над `peft + transformers + bitsandbytes`. Он даёт ~2× speedup и ~40% меньше VRAM на T4 за счёт ручных Triton-кернелей под attention. Альтернатива — голый PEFT, работает, но медленнее.

Версии пинуем — unsloth ломается при mismatch с transformers.

In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl<0.13.0" "datasets>=2.18" "transformers>=4.45" "accelerate>=0.34"

In [ ]:
# Проверка GPU + версий
import torch
import unsloth, transformers, trl, datasets

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0)
print(f"unsloth      = {unsloth.__version__}")
print(f"transformers = {transformers.__version__}")
print(f"trl          = {trl.__version__}")
print(f"datasets     = {datasets.__version__}")

## 1. Загрузка данных

Два варианта — выбери один:

**(A) Загрузить файлы вручную** через панель Files слева. Удобно для маленького датасета (<50 MB).

**(B) Подключить Google Drive.** Положи `train.jsonl` / `val.jsonl` в `MyDrive/text2sql_finetune/` и расшарь в Drive.

In [ ]:
# (B) Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/text2sql_finetune/*.jsonl /content/
!ls -la /content/*.jsonl

In [ ]:
# Sanity-check данных
import json
with open("train.jsonl") as f:
    sample = [json.loads(line) for line in f][:3]
for ex in sample:
    msgs = ex["messages"]
    print(f"--- example ({ex.get('_meta', {}).get('task')}) ---")
    for m in msgs:
        prev = m['content'][:120].replace('\n', ' ')
        print(f"  [{m['role']:9s}] {prev}...")
    print()

## 2. Загрузка базовой модели в 4-bit

**Учебный нюанс — память T4:**
- 4B параметров в fp16 = 8 GB. Уже впритык, без места на оптимизатор.
- 4B параметров в 4-bit (NF4 quantization) = ~2.5 GB.
- LoRA-адаптеры r=16: ~50 M обучаемых параметров. В fp16 = 100 MB. AdamW state = 200 MB.
- Активации при batch=2, seq=4096: ~3 GB.
- Итого ~6 GB peak — спокойно в 16 GB T4.

**Почему 4-bit, а не 8-bit:** базовая модель ЗАМОРОЖЕНА. Её точность не критична, потому что обучается только LoRA-«дельта». В 4-bit потери качества инференса <1%.

**Если Qwen3.5-4B недоступна:** замени `model_name` на `Qwen/Qwen2.5-4B` или `Qwen/Qwen3-4B-Base` — пайплайн не зависит от конкретной версии.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096  # ОБРЕЗАЕМ длинные системные промпты — на T4 больше не безопасно

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-4B",     # при необходимости замени на Qwen2.5-4B
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,                         # авто (bf16 на T4 не работает, упадёт в fp16)
    load_in_4bit=True,
)
print(f"loaded. tokenizer chat template length: {len(tokenizer.chat_template or '')}")

## 3. Wrap в LoRA

**Гиперпараметры — почему именно эти:**
- `r=16` — стандарт для 4B на специализированной задаче. Меньше r — недостаточная capacity, больше — переобучение и лишний VRAM.
- `lora_alpha=32` (= 2×r) — стандартное соотношение. Альфа отвечает за эффективный learning rate LoRA-веток.
- `target_modules` — все 7 проекций attention+MLP. Можно ограничиться только attention (`q,k,v,o`) — будет на 30% быстрее, но качество чуть ниже.
- `lora_dropout=0.05` — лёгкая регуляризация. На больших датасетах (>10k) можно 0.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # экономит VRAM ценой 20% времени
    random_state=42,
)
model.print_trainable_parameters()

## 4. Применяем chat template

**Учебный нюанс — почему `apply_chat_template`:**
Каждое семейство моделей имеет свой формат разделителей. Qwen использует ChatML:

```
<|im_start|>system
...
<|im_end|>
<|im_start|>user
...
<|im_end|>
<|im_start|>assistant
...
<|im_end|>
```

Если форматировать вручную — легко промахнуться (пропустить `\n`, перепутать токены) и модель учится мусору. `tokenizer.apply_chat_template` гарантированно даёт **тот же** строковой формат, что увидит `tokenizer.encode` на инференсе. Это та самая инвариантность train==inference.

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={
    "train": "train.jsonl",
    "val":   "val.jsonl",
})

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

ds = raw.map(format_chat, remove_columns=["messages", "_meta"])
print(f"train: {len(ds['train'])}, val: {len(ds['val'])}")
print("--- sample formatted text (first 500 chars) ---")
print(ds["train"][0]["text"][:500])

In [ ]:
# Сколько токенов получается? Это важно — выяснить, не уходим ли за MAX_SEQ_LEN.
import numpy as np
lens = [len(tokenizer.encode(x["text"])) for x in ds["train"].select(range(min(500, len(ds['train']))))]
print(f"token len: p50={np.percentile(lens,50):.0f}  p90={np.percentile(lens,90):.0f}  "
      f"p99={np.percentile(lens,99):.0f}  max={max(lens)}")
print(f"будут обрезаны: {sum(1 for l in lens if l > MAX_SEQ_LEN)} / {len(lens)} (sample)")
# Если >5% обрезается — увеличь MAX_SEQ_LEN до 6144 (если памяти хватает)
# или сократи db_schema/column_stats в системных промптах

## 5. Trainer

**Гиперпараметры обучения:**
- `lr=2e-4` — типично для LoRA. Full FT использует 1e-5; LoRA даёт «дельту» поверх замороженной модели → можно учить агрессивнее.
- `warmup_ratio=0.05` — линейный разгон в первых 5% шагов. Без warmup loss часто взрывается на первых батчах.
- `2 эпохи` — золотая середина. 1 эпоха часто недоучивает, 3+ переобучается на конкретных схемах.
- `effective batch = 8` — `per_device=2 × grad_accum=4`. На T4 нельзя поднять выше per_device=2 при seq=4096.
- `bf16=False` — T4 не поддерживает bfloat16, только fp16. На A100/H100 ставь `bf16=True`.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        output_dir="outputs/qwen35-4b-text2sql-lora",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=100,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        packing=False,        # не объединяем короткие примеры — сохраняет per-example loss
        report_to="none",
        seed=42,
    ),
)

In [ ]:
# Smoke-train: 50 шагов на маленьком сабсэмпле, чтобы убедиться, что ничего не падает.
# Если loss взрывается / NaN на этом этапе — что-то с chat-template или dtype.
import copy
smoke_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"].select(range(min(64, len(ds['train'])))),
    args=copy.deepcopy(trainer.args),
)
smoke_trainer.args.max_steps = 20
smoke_trainer.args.eval_strategy = "no"
smoke_trainer.args.save_strategy = "no"
smoke_trainer.train()
# Если loss падает с ~1.5-3 до ~0.5-1.5 за 20 шагов — всё нормально, можно полный train.
# Если NaN или растёт — проблема.

In [ ]:
# Полный прогон
trainer.train()

In [ ]:
# Сохраняем LoRA-адаптер (~150 MB, маленький)
model.save_pretrained("outputs/qwen35-4b-text2sql-lora/adapter")
tokenizer.save_pretrained("outputs/qwen35-4b-text2sql-lora/adapter")
!ls -la outputs/qwen35-4b-text2sql-lora/adapter/
!du -sh outputs/qwen35-4b-text2sql-lora/adapter/

## 6. Quick eval — посмотреть, как модель отвечает

Перед тем как мерджить и конвертировать — ручная проверка на 2-3 примерах из val. Если модель отвечает мусором или не следует формату — что-то не так с chat template / training format.

In [ ]:
FastLanguageModel.for_inference(model)  # переключает unsloth в режим инференса (ускорение)

import json
raw_val = [json.loads(l) for l in open("val.jsonl")][:3]

for ex in raw_val:
    msgs = ex["messages"][:2]  # без assistant
    inputs = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to("cuda")
    out = model.generate(inputs, max_new_tokens=256, do_sample=False, temperature=0.0)
    pred = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    gold = ex["messages"][2]["content"]
    print(f"--- {ex.get('_meta', {})} ---")
    print(f"PRED: {pred[:300]}")
    print(f"GOLD: {gold[:300]}\n")

## 7. Merge LoRA → HF safetensors → GGUF q4_K_M

**Учебный нюанс — формат GGUF:**
- `q4_K_M` — 4-bit с K-quants и Medium-блоками. Лучший trade-off для 4B: ~3 GB файл, потеря качества <1% относительно fp16.
- `q5_K_M` — 5-bit, ~3.5 GB, потери почти 0. Если место не критично — бери.
- `q8_0` — 8-bit, ~4.5 GB. Эталон, минимальные потери. Лучше для финальной публикации.
- `q3_K_S` — 3-bit, ~2 GB. Заметно деградирует на сложных SQL. Не бери для production.

Конвертация ~5 минут на 4B.

In [ ]:
# unsloth helper делает merge + GGUF в одной операции
model.save_pretrained_gguf(
    "qwen35-text2sql-lora",
    tokenizer,
    quantization_method="q4_k_m",
)
!ls -la qwen35-text2sql-lora/

In [ ]:
# Скачать GGUF локально (или сохранить в Drive)
!cp qwen35-text2sql-lora/*.gguf /content/drive/MyDrive/text2sql_finetune/ 2>/dev/null && echo 'saved to Drive' || echo 'no Drive'
# либо через File browser → Download

## 8. Что дальше

1. Скачай GGUF локально на Mac.
2. Положи в `~/.lmstudio/models/local/qwen35-text2sql/qwen35-text2sql.q4_k_m.gguf` (создай папку — LM Studio её увидит).
3. В LM Studio: открой модель → Local Server → Start.
4. В `.env` проекта переключи:
   ```
   LLM_BASE_URL=http://localhost:1234/v1
   LLM_API_KEY=lm-studio
   LLM_MODEL_NAME=qwen35-text2sql
   ```
5. `./scripts/run_ablation.sh feat/lora_finetune` — eval на bird_small + ambrosia_small.

См. `scripts/lm_studio_smoke_test.py` для проверки локального API.